# 03 STFT 与频谱图

1. STFT 核心计算与参数
2. 线性幅度与对数幅度
3. 相位、逆 STFT 与重建
4. 窗函数对比（Hann / Hamming / Blackman）
5. 时频权衡：窗长对时间定位与频率分辨的影响
6. 频谱图中的四种音乐结构形态


## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

## 2. STFT 核心计算

STFT：把信号切成短帧，对每帧加窗后做 FFT。区分实际窗长 $L$、FFT 点数 $N$ 与帧移 $H$：

$$X[k,t]=\sum_{n=0}^{L-1}x[n+tH]w[n]e^{-j2\pi kn/N},\qquad L\leq N.$$

- $L$ = `win_length`：实际观察时长
- $N$ = `n_fft`：DFT 点数和频率采样网格
- $H$ = `hop_length`：帧移

实值音频只保留非负频率时，输出形状为 `(1 + floor(n_fft/2), n_frames)`。当 `n_fft > win_length` 时，librosa 将较短窗在左右两侧居中补零；这会加密频率采样，但不会增加由窗长决定的真实分辨能力。

上式采用未居中分帧（`center=False`）的索引写法。本 Notebook 的 librosa 调用沿用默认 `center=True`：波形先在两端填充 `n_fft//2`，第 $t$ 帧中心对齐到 $tH$；因此公式索引与库函数的边界帧不能在忽略填充后逐项等同。


In [ ]:
def load_and_stft(path, start_sec=0.0, duration_sec=3.0,
                  n_fft=2048, win_length=None, hop_length=512):
    """加载音频并计算 STFT、幅度谱与相对 dB 频谱图。"""
    if win_length is None:
        win_length = n_fft
    samples, sr = librosa.load(
        path, sr=SAMPLE_RATE, mono=True, offset=start_sec, duration=duration_sec
    )
    stft_complex = librosa.stft(
        samples, n_fft=n_fft, win_length=win_length,
        hop_length=hop_length, window="hann"
    )
    magnitude = np.abs(stft_complex)
    log_spectrogram = librosa.amplitude_to_db(magnitude, ref=np.max)
    return samples, sr, magnitude, log_spectrogram

piano_path = DATASET_DIR / "piano_solo.wav"
piano_samples, sr, piano_mag, piano_logspec = load_and_stft(
    piano_path, start_sec=10.0, duration_sec=3.0,
    n_fft=2048, win_length=2048, hop_length=512
)
print(f"钢琴片段：{len(piano_samples)} 采样点 @ {sr} Hz")
print(f"STFT 输出形状：{piano_mag.shape} (频率 bins × 时间帧)")
print(f"DFT bin 间距：{SAMPLE_RATE / 2048:.2f} Hz/bin")
print(f"时间步长：{512 / SAMPLE_RATE * 1000:.2f} ms/frame")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左图：线性幅度频谱图，强分量占据大部分显示范围
img0 = librosa.display.specshow(
    piano_mag, sr=SAMPLE_RATE, hop_length=512, x_axis="time", y_axis="log",
    ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("线性幅度频谱图")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylim(50, SAMPLE_RATE // 2)
fig.colorbar(img0, ax=axes[0], format="%+2.0f")

# 右图：对数幅度频谱图，压缩显示动态范围
img1 = librosa.display.specshow(
    piano_logspec, sr=SAMPLE_RATE, hop_length=512, x_axis="time", y_axis="log",
    ax=axes[1], cmap="Greys_r"
)
axes[1].set_title("对数幅度频谱图（dB）")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylim(50, SAMPLE_RATE // 2)
fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

plt.suptitle("为何使用对数尺度：钢琴独奏", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "stft_linear_vs_log.png", dpi=600, bbox_inches="tight")
plt.show()

左图的线性幅度动态范围很大，弱谐波难以看清；右图用相对 dB 压缩动态范围。普通 log-magnitude 是可视化和建模变换，不包含响度模型所需的频率加权、时间整合与门限，不能直接等同于“人耳听到的响度”。


## 3. 相位、逆 STFT 与重建

STFT 的输出是复数：幅度表示各频率分量的强弱，相位表示这些分量在当前帧中的周期位置。

如果保留复数 STFT，`librosa.istft` 可以在合适的窗函数与重叠条件下几乎无损地重建波形。只保留幅度谱时，相位已经丢失，只能用 Griffin-Lim 这类迭代算法近似估计。


In [ ]:
# 使用前面加载的钢琴片段，比较三种重建方式。
N_FFT = 2048
HOP_LENGTH = 512

stft_complex = librosa.stft(piano_samples, n_fft=N_FFT, hop_length=HOP_LENGTH, window="hann")
magnitude = np.abs(stft_complex)

reconstructed_with_phase = librosa.istft(
    stft_complex, hop_length=HOP_LENGTH, window="hann", length=len(piano_samples)
)

rng = np.random.default_rng(0)
random_phase = np.exp(1j * rng.uniform(-np.pi, np.pi, size=magnitude.shape))
reconstructed_random_phase = librosa.istft(
    magnitude * random_phase, hop_length=HOP_LENGTH, window="hann", length=len(piano_samples)
)

reconstructed_griffinlim = librosa.griffinlim(
    magnitude,
    n_iter=32,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=N_FFT,
    window="hann",
    length=len(piano_samples),
    random_state=0,
)


def reconstruction_snr(reference, estimate):
    error = reference - estimate
    return 10 * np.log10((np.mean(reference ** 2) + 1e-12) / (np.mean(error ** 2) + 1e-12))

start = int(0.40 * sr)
length = int(0.12 * sr)
time_ms = np.arange(length) / sr * 1000

series = [
    (piano_samples, "原始波形"),
    (reconstructed_with_phase, f"保留复数 STFT → istft（SNR {reconstruction_snr(piano_samples, reconstructed_with_phase):.1f} dB）"),
    (reconstructed_griffinlim, f"仅幅度谱 → Griffin-Lim（SNR {reconstruction_snr(piano_samples, reconstructed_griffinlim):.1f} dB）"),
    (reconstructed_random_phase, f"幅度谱 + 随机相位（SNR {reconstruction_snr(piano_samples, reconstructed_random_phase):.1f} dB）"),
]

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True, sharey=True)
for ax, (samples, title) in zip(axes, series):
    ax.plot(time_ms, samples[start:start + length], color="0.10", lw=0.9)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_ylabel("振幅")
    ax.axhline(0, color="0.7", lw=0.5)
axes[-1].set_xlabel("时间 (ms，相对截取片段)")

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "phase_reconstruction_comparison.png", dpi=600, bbox_inches="tight")
plt.show()


**观察**：保留复数 STFT 时，重建结果与原始波形几乎重合。随机相位重建与原波形差异明显；Griffin--Lim 能从幅度谱中恢复可听的近似波形，但无法保证与原始相位一致。


## 4. 窗函数对比

截断波形做 FFT 等价于「周期化」，帧边界的不连续会导致**频谱泄漏**。
窗函数降低帧边界处的权重，减小周期延拓时的边界不连续，从而削弱泄漏。

三种常见窗：

- **Hann**：两端降到零，旁瓣适中，librosa 默认
- **Hamming**：保留少量端点权重，第一旁瓣更低
- **Blackman**：旁瓣压得更低，但主瓣变宽、频率分辨率变差

下面用一个偏离 DFT bin 中心的合成正弦帧，在相同条件下比较不同窗对频谱泄漏的抑制效果。

In [ ]:
# 用离开 DFT bin 中心的单一正弦隔离窗函数造成的谱泄漏
frame_length = 2048
test_frequency_hz = 440.37
frame_time = np.arange(frame_length) / SAMPLE_RATE
frame = np.sin(2 * np.pi * test_frequency_hz * frame_time)

windows = {
    "Hann": np.hanning(frame_length),
    "Hamming": np.hamming(frame_length),
    "Blackman": np.blackman(frame_length),
    "Rectangular（无窗）": np.ones(frame_length),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (name, win) in zip(axes, windows.items()):
    # 加窗后做 FFT
    windowed = frame * win
    spectrum = np.fft.rfft(windowed)
    magnitude_db = 20 * np.log10(np.abs(spectrum) / np.max(np.abs(spectrum)) + 1e-10)
    freq_axis = np.fft.rfftfreq(frame_length, 1 / SAMPLE_RATE)
    
    ax.plot(freq_axis, magnitude_db, lw=1.0, color="0.1")
    ax.set_xlim(0, 2000)  # 只看前 2000 Hz
    ax.set_ylim(-120, 5)
    ax.set_title(name)
    ax.set_xlabel("频率 (Hz)")
    ax.set_ylabel("幅度 (dB)")
    ax.axhline(-40, color="red", ls="--", lw=0.5, alpha=0.5, label="-40 dB")
    ax.axhline(-80, color="red", ls="--", lw=0.5, alpha=0.3, label="-80 dB")

plt.suptitle(f"窗函数对谱泄漏的影响（{test_frequency_hz:.2f} Hz，偏离 DFT bin 中心）", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "window_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

**观察**：对这个离开 DFT bin 中心的单一正弦，矩形窗旁瓣最高；Blackman 的旁瓣更低，但主瓣更宽。不同窗的精确旁瓣高度与主瓣宽度应在相同归一化、FFT 长度和周期/对称窗约定下比较。

librosa 默认的 Hann 窗是常见起点；重建、测量或特定检测任务仍应按重叠、旁瓣和主瓣需求验证。

## 5. 时频权衡

当 `win_length = n_fft = N` 时，窗越长，分辨相近频率的能力通常越强，但快速变化会在窗内被平均。下面固定 `hop_length=512`，只改变窗长/FFT 点数，避免把帧密度变化混入对比：

- `n_fft=512`（约 23 ms）
- `n_fft=2048`（约 93 ms）
- `n_fft=4096`（约 186 ms）


In [ ]:
n_fft_values = [512, 2048, 4096]
hop_length = 512  # 固定时间采样步长，隔离窗长/FFT 点数的影响

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, n_fft in zip(axes, n_fft_values):
    stft_complex = librosa.stft(
        piano_samples, n_fft=n_fft, win_length=n_fft,
        hop_length=hop_length, window="hann"
    )
    log_spec = librosa.amplitude_to_db(np.abs(stft_complex), ref=np.max)
    img = librosa.display.specshow(
        log_spec, sr=SAMPLE_RATE, hop_length=hop_length,
        x_axis="time", y_axis="log", ax=ax, cmap="Greys_r"
    )
    ax.set_title(f"win_length=n_fft={n_fft}（{n_fft/SAMPLE_RATE*1000:.0f} ms）")
    ax.set_xlabel("时间 (s)")
    ax.set_ylim(50, SAMPLE_RATE // 2)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

plt.suptitle("时频权衡：固定 hop_length=512，只改变窗长与 FFT 点数", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "time_frequency_tradeoff.png", dpi=600, bbox_inches="tight")
plt.show()


**观察口径**：固定 `hop_length` 后，各图横轴帧密度一致。短窗更容易定位快速变化，长窗更容易分开相近频率；具体清晰度仍取决于当前片段内容、窗函数和显示尺度。


## 6. 频谱图中的四种音乐结构形态

部分音乐结构会在频谱图中形成可辨认的几何形状。

| 形态 | 来源 | 本章素材 |
|:---|:---|:---|
| **水平条纹** | 持续音（基频+谐波） | 小提琴长音 |
| **垂直条纹** | 宽带瞬态（打击乐、辅音） | 打击乐片段 |
| **平行条纹组** | 和弦 / 多声部 | 钢琴和弦 |
| **斜线或弯曲轨迹** | 滑音 / 连续音高变化 | 人声旋律 |

下图比较四段录音中的这些形态。

In [ ]:
# 定义四段示范音频及其切片参数
shape_examples = {
    "水平条纹（持续音）": {
        "path": DATASET_DIR / "zhao_violin_wet.wav",
        "start": 15.0,
        "duration": 4.0,
        "description": "小提琴长音：基频和多次谐波呈水平亮带",
    },
    "垂直条纹（瞬态）": {
        "path": DATASET_DIR / "orch_perc.wav",
        "start": 2.0,
        "duration": 3.0,
        "description": "打击乐：宽带瞬态在较宽频率范围内形成垂直亮带",
    },
    "平行条纹（和弦 / 复音）": {
        "path": DATASET_DIR / "piano_solo.wav",
        "start": 12.0,
        "duration": 3.0,
        "description": "钢琴和弦：多个音各自贡献一组水平条纹，交织成平行组",
    },
    "斜线或弯曲轨迹（连续音高变化）": {
        "path": DATASET_DIR / "xiaohetang_vox.wav",
        "start": 32.0,
        "duration": 4.0,
        "description": "人声：基频随旋律连续变化，形成倾斜或弯曲轨迹",
    },
}

# 预计算各段的频谱图
spec_data = {}
for label, info in shape_examples.items():
    samples, sr = librosa.load(info["path"], sr=SAMPLE_RATE, mono=True, offset=info["start"], duration=info["duration"])
    stft = librosa.stft(samples, n_fft=2048, hop_length=512, window="hann")
    log_spec = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    spec_data[label] = {
        "samples": samples,
        "log_spec": log_spec,
        "description": info["description"],
    }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (label, data) in zip(axes, spec_data.items()):
    img = librosa.display.specshow(
        data["log_spec"], sr=SAMPLE_RATE, hop_length=512,
        x_axis="time", y_axis="log", ax=ax, cmap="Greys_r"
    )
    ax.set_title(label + "\n" + data["description"], fontsize=10)
    ax.set_xlabel("时间 (s)")
    ax.set_ylim(50, SAMPLE_RATE // 2)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

plt.suptitle("频谱图中的四种几何形态", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "spectrogram_shapes.png", dpi=600, bbox_inches="tight")
plt.show()

## 7. 小结与参数速查

| 参数 | 符号 | librosa 默认 | 含义 |
|:---|:---:|:---|:---|
| `n_fft` | $N$ | 2048 | FFT 点数与输出频率 bin 数 |
| `win_length` | $L$ | `n_fft` | 实际窗长，主导时间观察范围与真实频率分辨能力 |
| `hop_length` | $H$ | `n_fft // 4` | 相邻帧中心的时间步长 |
| `window` | $w[n]$ | `'hann'` | 窗函数类型 |

当 `n_fft > win_length` 时，额外零填充只加密 DFT 网格。讨论时频权衡时应明确究竟改变了窗长、FFT 点数，还是两者同时改变。

下一步（`04_mel_cqt_mfcc_chroma.ipynb`）将重新组织频率轴：Mel、CQT、MFCC 与 Chroma。


In [ ]:
print("本 Notebook 生成的图像文件：")
generated_names = [
    "stft_linear_vs_log.png", "phase_reconstruction_comparison.png",
    "window_comparison.png", "time_frequency_tradeoff.png",
    "spectrogram_shapes.png",
]
for f in [OUTPUT_FIG_DIR / name for name in generated_names]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:8.1f} KB")